In [ ]:
"""
build_market_dataset.py

全市场量化数据构建器 v1

功能：
1. 自动从 BaoStock 获取 A 股股票池
2. 支持扫描：
   - all      = 全 A 股
   - growth   = 300 + 688
   - custom   = 自定义股票池
3. 自动过滤：
   - ST
   - 停牌
   - 数据太少
   - 成交额过低
4. 生成特征：
   - 均线
   - 量比
   - RSI
   - MACD
   - 布林带
   - 彩虹均线评分
   - 主力行为评分
5. 自动打标签 final_label
6. 输出：
   output/a_share_labeled_dataset_with_auto_manual_label.csv

安装：
    pip install baostock pandas numpy

运行：
    python3 build_market_dataset.py

建议：
    第一次先用 growth 模式，也就是 300 + 688。
"""

import os
import time
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import baostock as bs
except ImportError:
    bs = None


# =========================
# 配置区
# =========================

OUTPUT_DIR = "output"
OUTPUT_PATH = "output/a_share_labeled_dataset_with_auto_manual_label.csv"
STOCK_POOL_PATH = "output/stock_pool.csv"
FAILED_PATH = "output/failed_symbols.csv"

START_DATE = "2024-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

# 可选："growth" = 300 + 688；"all" = 全 A 股；"custom" = 自定义股票池
MARKET_MODE = "growth"

# 首次测试建议设小一点，例如 100。确认能跑通后改成 None。
MAX_STOCKS = 300

SLEEP_BETWEEN_STOCKS = 0.3

FUTURE_DAYS = 5
UP_THRESHOLD = 0.05
DOWN_THRESHOLD = -0.03

MIN_HISTORY_ROWS = 180
MIN_AMOUNT_MEAN = 80_000_000  # 最近20日平均成交额低于8000万的过滤掉

CUSTOM_SYMBOLS = [
    "sh.600519",
    "sz.300750",
    "sh.688012",
]


# =========================
# 工具函数
# =========================

def ensure_baostock():
    if bs is None:
        raise ImportError("请先安装 baostock：pip install baostock")


def login_baostock():
    ensure_baostock()
    print("登录 BaoStock...")
    result = bs.login()
    if result.error_code != "0":
        raise RuntimeError(f"BaoStock 登录失败：{result.error_msg}")


def logout_baostock():
    try:
        bs.logout()
        print("BaoStock 已登出。")
    except Exception:
        pass


def normalize_symbol_for_display(code: str) -> str:
    return code.replace("sh.", "").replace("sz.", "")


def get_stock_pool() -> list[str]:
    """
    获取股票池。
    BaoStock 的 query_stock_basic 返回股票基础信息。
    """
    if MARKET_MODE == "custom":
        return CUSTOM_SYMBOLS

    print("获取股票池...")

    rs = bs.query_stock_basic()
    if rs.error_code != "0":
        raise RuntimeError(f"获取股票池失败：{rs.error_msg}")

    rows = []
    while rs.next():
        rows.append(rs.get_row_data())

    stock_df = pd.DataFrame(rows, columns=rs.fields)

    if stock_df.empty:
        raise ValueError("股票池为空。")

    # 常见字段：code, code_name, ipoDate, outDate, type, status
    stock_df = stock_df.copy()

    if "type" in stock_df.columns:
        # type=1 通常是股票，过滤指数等
        stock_df = stock_df[stock_df["type"] == "1"]

    if "status" in stock_df.columns:
        # status=1 通常是上市
        stock_df = stock_df[stock_df["status"] == "1"]

    stock_df["code"] = stock_df["code"].astype(str)

    if MARKET_MODE == "growth":
        stock_df = stock_df[
            stock_df["code"].str.startswith("sz.300")
            | stock_df["code"].str.startswith("sh.688")
        ]

    elif MARKET_MODE == "all":
        stock_df = stock_df[
            stock_df["code"].str.startswith("sh.6")
            | stock_df["code"].str.startswith("sz.0")
            | stock_df["code"].str.startswith("sz.3")
        ]

    else:
        raise ValueError("MARKET_MODE 只能是 growth / all / custom")

    if MAX_STOCKS is not None:
        stock_df = stock_df.head(MAX_STOCKS)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    stock_df.to_csv(STOCK_POOL_PATH, index=False, encoding="utf-8-sig")

    symbols = stock_df["code"].tolist()

    print(f"股票池数量：{len(symbols)}")
    print(f"股票池已保存：{STOCK_POOL_PATH}")

    return symbols


# =========================
# 数据获取
# =========================

def fetch_stock_data(symbol: str) -> pd.DataFrame:
    fields = (
        "date,code,open,high,low,close,preclose,volume,amount,"
        "adjustflag,turn,tradestatus,pctChg,isST"
    )

    rs = bs.query_history_k_data_plus(
        symbol,
        fields,
        start_date=START_DATE,
        end_date=END_DATE,
        frequency="d",
        adjustflag="2",
    )

    if rs.error_code != "0":
        raise RuntimeError(f"BaoStock错误：{rs.error_msg}")

    rows = []
    while rs.next():
        rows.append(rs.get_row_data())

    df = pd.DataFrame(rows, columns=rs.fields)

    if df.empty:
        raise ValueError("空数据")

    df = df.rename(
        columns={
            "code": "symbol",
            "turn": "turnover",
            "pctChg": "pct_change",
        }
    )

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    numeric_cols = [
        "open", "high", "low", "close", "preclose",
        "volume", "amount", "turnover", "pct_change",
        "tradestatus", "isST",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[df["tradestatus"] == 1].copy()
    df = df[df["isST"] == 0].copy()

    if len(df) < MIN_HISTORY_ROWS:
        raise ValueError(f"历史数据太少：{len(df)}")

    recent_amount_mean = df["amount"].tail(20).mean()
    if pd.isna(recent_amount_mean) or recent_amount_mean < MIN_AMOUNT_MEAN:
        raise ValueError(f"成交额过低：{recent_amount_mean:.0f}")

    df["change"] = df["close"] - df["preclose"]
    df["amplitude"] = (df["high"] - df["low"]) / df["preclose"] * 100

    df["symbol"] = df["symbol"].str.replace("sh.", "", regex=False)
    df["symbol"] = df["symbol"].str.replace("sz.", "", regex=False)

    df = df.sort_values("date").reset_index(drop=True)

    return df


# =========================
# 特征工程
# =========================

def add_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    g["return_1d"] = g["close"].pct_change()

    for ma in [5, 10, 20, 60, 120]:
        g[f"ma{ma}"] = g["close"].rolling(ma).mean()

    g["close_ma5_ratio"] = g["close"] / g["ma5"] - 1
    g["close_ma20_ratio"] = g["close"] / g["ma20"] - 1
    g["close_ma60_ratio"] = g["close"] / g["ma60"] - 1

    g["vol_ma5"] = g["volume"].rolling(5).mean()
    g["vol_ma20"] = g["volume"].rolling(20).mean()
    g["volume_ratio"] = g["volume"] / g["vol_ma20"]

    g["volatility_10d"] = g["return_1d"].rolling(10).std()
    g["volatility_20d"] = g["return_1d"].rolling(20).std()

    g["body_ratio"] = (g["close"] - g["open"]) / g["open"]
    g["upper_shadow_ratio"] = (
        g["high"] - g[["open", "close"]].max(axis=1)
    ) / g["open"]
    g["lower_shadow_ratio"] = (
        g[["open", "close"]].min(axis=1) - g["low"]
    ) / g["open"]

    g["high_20d"] = g["high"].rolling(20).max()
    g["low_20d"] = g["low"].rolling(20).min()

    # RSI
    delta = g["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain_14 = gain.rolling(14).mean()
    avg_loss_14 = loss.rolling(14).mean()
    rs_14 = avg_gain_14 / avg_loss_14
    g["rsi_14"] = 100 - (100 / (1 + rs_14))

    avg_gain_6 = gain.rolling(6).mean()
    avg_loss_6 = loss.rolling(6).mean()
    rs_6 = avg_gain_6 / avg_loss_6
    g["rsi_6"] = 100 - (100 / (1 + rs_6))

    # MACD
    ema12 = g["close"].ewm(span=12, adjust=False).mean()
    ema26 = g["close"].ewm(span=26, adjust=False).mean()

    g["macd"] = ema12 - ema26
    g["macd_signal"] = g["macd"].ewm(span=9, adjust=False).mean()
    g["macd_diff"] = g["macd"] - g["macd_signal"]

    # Bollinger
    mid = g["close"].rolling(20).mean()
    std = g["close"].rolling(20).std()

    g["boll_high"] = mid + 2 * std
    g["boll_low"] = mid - 2 * std
    g["boll_width"] = (g["boll_high"] - g["boll_low"]) / g["close"]
    g["boll_position"] = (
        (g["close"] - g["boll_low"]) / (g["boll_high"] - g["boll_low"])
    )

    return g


def add_rainbow_score(g: pd.DataFrame) -> pd.DataFrame:
    scores = []

    for _, row in g.iterrows():
        s = 0

        needed = [row.get("ma5"), row.get("ma10"), row.get("ma20"), row.get("ma60")]
        if any(pd.isna(x) for x in needed):
            scores.append(np.nan)
            continue

        if row["ma5"] > row["ma10"] > row["ma20"] > row["ma60"]:
            s += 40

        if row["close"] > row["ma20"]:
            s += 20

        if row["close"] > row["ma60"]:
            s += 20

        spread = row["ma5"] / row["ma60"] - 1

        if spread > 0.05:
            s += 20

        scores.append(min(s, 100))

    g["rainbow_score"] = scores
    return g


def add_main_force_score(g: pd.DataFrame) -> pd.DataFrame:
    scores = []
    signals = []

    for _, row in g.iterrows():
        s = 0
        signal_list = []

        volume_ratio = row.get("volume_ratio")
        if pd.isna(volume_ratio):
            scores.append(np.nan)
            signals.append("")
            continue

        if volume_ratio > 2:
            s += 30
            signal_list.append("放量")

        if row["pct_change"] > 5:
            s += 30
            signal_list.append("大涨")

        if row["turnover"] > 5:
            s += 20
            signal_list.append("换手活跃")

        if row["close"] > row["open"]:
            s += 20
            signal_list.append("阳线")

        if row.get("break_20d_high", 0) == 1:
            s += 20
            signal_list.append("突破20日高点")

        scores.append(min(s, 100))
        signals.append("；".join(signal_list))

    g["main_force_score"] = scores
    g["main_force_signal"] = signals

    return g


def add_extra_short_term_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()

    g["high_10d"] = g["high"].rolling(10).max()
    g["high_20d_prev"] = g["high"].rolling(20).max().shift(1)
    g["break_20d_high"] = (g["close"] >= g["high_20d_prev"] * 0.99).astype(int)

    g["pct_change_3d"] = g["close"].pct_change(3) * 100
    g["pct_change_5d"] = g["close"].pct_change(5) * 100
    g["amount_ma20"] = g["amount"].rolling(20).mean()

    return g


# =========================
# 标签
# =========================

def add_labels(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()

    g["future_close"] = g["close"].shift(-FUTURE_DAYS)
    g["future_return"] = g["future_close"] / g["close"] - 1

    g["label"] = 0
    g.loc[g["future_return"] >= UP_THRESHOLD, "label"] = 1
    g.loc[g["future_return"] <= DOWN_THRESHOLD, "label"] = -1

    def auto_manual_label(row):
        rainbow_score = row.get("rainbow_score", 0)
        main_force_score = row.get("main_force_score", 0)
        pct_change = row.get("pct_change", 0)
        future_return = row.get("future_return", 0)

        if pd.isna(rainbow_score):
            rainbow_score = 0
        if pd.isna(main_force_score):
            main_force_score = 0
        if pd.isna(pct_change):
            pct_change = 0
        if pd.isna(future_return):
            future_return = 0

        if pct_change >= 5 and future_return < 0:
            return -1

        if future_return <= DOWN_THRESHOLD:
            return -1

        if rainbow_score >= 80 and main_force_score >= 60:
            return 1

        if pct_change >= 5 and main_force_score >= 60 and future_return > 0:
            return 1

        return 0

    g["manual_label"] = g.apply(auto_manual_label, axis=1)
    g["final_label"] = g["manual_label"]

    return g


# =========================
# 单只股票处理
# =========================

def process_symbol(symbol: str) -> pd.DataFrame:
    g = fetch_stock_data(symbol)
    g = add_features(g)
    g = add_extra_short_term_features(g)
    g = add_rainbow_score(g)
    g = add_main_force_score(g)
    g = add_labels(g)

    g = g.dropna(subset=["future_return", "rainbow_score", "main_force_score"])

    if g.empty:
        raise ValueError("清洗后为空")

    return g


# =========================
# 主流程
# =========================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    login_baostock()

    all_data = []
    failed_records = []

    try:
        symbols = get_stock_pool()

        for idx, symbol in enumerate(symbols, start=1):
            print(f"\n[{idx}/{len(symbols)}] 处理 {symbol}")

            try:
                g = process_symbol(symbol)
                all_data.append(g)
                print(f"{symbol} 完成，样本数：{len(g)}")

            except Exception as error:
                failed_records.append(
                    {
                        "symbol": symbol,
                        "reason": str(error),
                    }
                )
                print(f"{symbol} 失败：{error}")

            time.sleep(SLEEP_BETWEEN_STOCKS)

    finally:
        logout_baostock()

    if failed_records:
        pd.DataFrame(failed_records).to_csv(
            FAILED_PATH,
            index=False,
            encoding="utf-8-sig",
        )
        print(f"\n失败记录已保存：{FAILED_PATH}")

    if not all_data:
        raise ValueError("没有任何股票成功，无法生成数据集。")

    dataset = pd.concat(all_data, ignore_index=True)
    dataset = dataset.sort_values(["date", "symbol"]).reset_index(drop=True)

    dataset.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print("\n==============================")
    print("全市场数据集构建完成")
    print(f"模式：{MARKET_MODE}")
    print(f"数据集路径：{OUTPUT_PATH}")
    print(f"总样本数：{len(dataset)}")
    print(f"股票数：{dataset['symbol'].nunique()}")
    print(f"日期范围：{dataset['date'].min()} -> {dataset['date'].max()}")

    print("\nfinal_label 分布：")
    print(dataset["final_label"].value_counts())
    print(dataset["final_label"].value_counts(normalize=True))

    print("\n下一步运行：")
    print("python3 train_model.py")
    print("python3 yaogu_radar_v2.py")
    print("streamlit run app.py")


if __name__ == "__main__":
    main()


In [ ]:
"""
train_model.py

功能：
1. 读取标注数据
2. 清洗数据
3. 训练随机森林模型
4. 输出模型 + 特征重要性
5. 生成 latest_stock_signals.csv

运行：
    python3 train_model.py
"""

import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import joblib

DATA_PATH = "output/a_share_labeled_dataset_with_auto_manual_label.csv"
MODEL_PATH = "output/a_share_random_forest_model.joblib"
SIGNAL_PATH = "output/latest_stock_signals.csv"

TEST_RATIO = 0.2


def load_data():
    print("📥 读取数据...")
    df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

    df["symbol"] = df["symbol"].astype(str).str.zfill(6)
    df["date"] = pd.to_datetime(df["date"])

    print("原始数据量：", len(df))
    return df


def clean_data(df):
    print("🧹 清洗数据...")

    df = df.dropna()

    # 只保留有意义标签
    df = df[df["final_label"].isin([-1, 0, 1])]

    print("清洗后数据量：", len(df))
    print("标签分布：")
    print(df["final_label"].value_counts())

    return df


def split_data(df):
    print("📊 切分训练集 / 测试集...")

    df = df.sort_values("date")

    split_index = int(len(df) * (1 - TEST_RATIO))

    train_df = df.iloc[:split_index]
    test_df = df.iloc[split_index:]

    print("训练集：", len(train_df))
    print("测试集：", len(test_df))

    return train_df, test_df


def get_features(df):
    ignore_cols = [
        "symbol", "date",
        "future_close", "future_return",
        "label", "manual_label", "final_label"
    ]

    features = [col for col in df.columns if col not in ignore_cols]

    return features


def train_model(train_df, features):
    print("🤖 训练模型...")

    X = train_df[features]
    y = train_df["final_label"]

    model = RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_split=50,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X, y)

    print("✅ 模型训练完成")

    return model


def evaluate_model(model, test_df, features):
    print("📈 简单评估...")

    X_test = test_df[features]
    y_test = test_df["final_label"]

    pred = model.predict(X_test)

    acc = (pred == y_test).mean()

    print("测试集准确率：", round(acc, 4))


def save_model(model):
    os.makedirs("output", exist_ok=True)
    joblib.dump(model, MODEL_PATH)
    print("💾 模型已保存：", MODEL_PATH)


def generate_signals(model, df, features):
    print("🚀 生成最新信号...")

    latest_date = df["date"].max()
    latest_df = df[df["date"] == latest_date].copy()

    X = latest_df[features]

    probs = model.predict_proba(X)

    latest_df["prob_down"] = probs[:, 0]
    latest_df["prob_mid"] = probs[:, 1]
    latest_df["prob_up"] = probs[:, 2]

    latest_df["pred_label"] = model.predict(X)

    latest_df["model_score"] = latest_df["prob_up"] * 100

    latest_df = latest_df.sort_values("model_score", ascending=False)

    output_cols = [
        "symbol", "date", "close", "pct_change",
        "rainbow_score", "main_force_score",
        "pred_label", "prob_up", "prob_down",
        "model_score"
    ]

    latest_df[output_cols].to_csv(
        SIGNAL_PATH, index=False, encoding="utf-8-sig"
    )

    print("📊 最新信号已生成：", SIGNAL_PATH)


def main():
    df = load_data()
    df = clean_data(df)

    train_df, test_df = split_data(df)

    features = get_features(df)

    model = train_model(train_df, features)

    evaluate_model(model, test_df, features)

    save_model(model)

    generate_signals(model, df, features)

    print("\n🎯 完成：模型 + 信号 已更新")


if __name__ == "__main__":
    main()

In [ ]:
"""
yaogu_model_v3.py

妖股模型 v3：短线实战版

核心思想：
1. 不追求每天都有票
2. 只筛“趋势 + 主力行为 + 短线结构 + 风险不过高”的票
3. 输出更干净的短线候选池

输入：
    output/a_share_labeled_dataset_with_auto_manual_label.csv
    output/latest_stock_signals.csv

输出：
    output/yaogu_model_v3_signals.csv

运行：
    python3 yaogu_model_v3.py
"""

import os
import numpy as np
import pandas as pd


DATASET_PATH = "output/a_share_labeled_dataset_with_auto_manual_label.csv"
LATEST_SIGNAL_PATH = "output/latest_stock_signals.csv"
OUTPUT_PATH = "output/yaogu_model_v3_signals.csv"


def safe_num(value, default=0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def add_short_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    g["high_10d"] = g["high"].rolling(10).max()
    g["high_20d"] = g["high"].rolling(20).max()
    g["high_60d"] = g["high"].rolling(60).max()

    g["low_20d"] = g["low"].rolling(20).min()
    g["low_60d"] = g["low"].rolling(60).min()

    g["amount_ma20"] = g["amount"].rolling(20).mean()
    g["volume_ma20"] = g["volume"].rolling(20).mean()
    g["volume_ratio_v3"] = g["volume"] / g["volume_ma20"]

    g["pct_3d"] = g["close"].pct_change(3) * 100
    g["pct_5d"] = g["close"].pct_change(5) * 100
    g["pct_10d"] = g["close"].pct_change(10) * 100
    g["pct_20d"] = g["close"].pct_change(20) * 100

    g["is_red"] = (g["close"] > g["open"]).astype(int)
    g["red_count_3d"] = g["is_red"].rolling(3).sum()
    g["red_count_5d"] = g["is_red"].rolling(5).sum()

    g["is_big_up"] = (g["pct_change"] >= 5).astype(int)
    g["is_limit_like"] = (g["pct_change"] >= 9.5).astype(int)
    g["big_up_count_10d"] = g["is_big_up"].rolling(10).sum()
    g["limit_like_count_20d"] = g["is_limit_like"].rolling(20).sum()

    g["break_20d_high"] = (g["close"] >= g["high_20d"].shift(1) * 0.99).astype(int)
    g["break_60d_high"] = (g["close"] >= g["high_60d"].shift(1) * 0.99).astype(int)

    g["position_60d"] = (g["close"] - g["low_60d"]) / (g["high_60d"] - g["low_60d"])

    g["upper_shadow_ratio_v3"] = (
        g["high"] - g[["open", "close"]].max(axis=1)
    ) / g["open"]

    g["lower_shadow_ratio_v3"] = (
        g[["open", "close"]].min(axis=1) - g["low"]
    ) / g["open"]

    g["real_body_ratio_v3"] = (g["close"] - g["open"]).abs() / g["open"]

    # 缩量回踩：靠近20日线，同时量比不高
    if "ma20" in g.columns:
        g["near_ma20"] = ((g["close"] / g["ma20"] - 1).abs() <= 0.03).astype(int)
    else:
        g["near_ma20"] = 0

    g["shrink_pullback"] = (
        (g["near_ma20"] == 1)
        & (g["volume_ratio_v3"] <= 0.8)
        & (g["close"] > g["open"])
    ).astype(int)

    return g


def score_yaogu_v3(row: pd.Series) -> pd.Series:
    score = 0
    risk = 0
    reasons = []

    rainbow = safe_num(row.get("rainbow_score"))
    main_force = safe_num(row.get("main_force_score"))
    model_score = safe_num(row.get("model_score"))
    prob_up = safe_num(row.get("prob_up"))
    prob_down = safe_num(row.get("prob_down"))
    pred_label = safe_num(row.get("pred_label"))

    pct_change = safe_num(row.get("pct_change"))
    pct_5d = safe_num(row.get("pct_5d"))
    pct_10d = safe_num(row.get("pct_10d"))
    pct_20d = safe_num(row.get("pct_20d"))

    turnover = safe_num(row.get("turnover"))
    volume_ratio = safe_num(row.get("volume_ratio_v3"))
    amount_ma20 = safe_num(row.get("amount_ma20"))

    red_count_3d = safe_num(row.get("red_count_3d"))
    red_count_5d = safe_num(row.get("red_count_5d"))
    big_up_count_10d = safe_num(row.get("big_up_count_10d"))
    limit_like_count_20d = safe_num(row.get("limit_like_count_20d"))

    break_20d = safe_num(row.get("break_20d_high"))
    break_60d = safe_num(row.get("break_60d_high"))
    position_60d = safe_num(row.get("position_60d"))

    upper_shadow = safe_num(row.get("upper_shadow_ratio_v3"))
    lower_shadow = safe_num(row.get("lower_shadow_ratio_v3"))
    shrink_pullback = safe_num(row.get("shrink_pullback"))

    # =========================
    # 1. 趋势基础
    # =========================

    if rainbow >= 90:
        score += 18
        reasons.append("彩虹趋势极强")
    elif rainbow >= 70:
        score += 12
        reasons.append("彩虹趋势较强")
    elif rainbow >= 50:
        score += 6
        reasons.append("趋势初步转强")

    # =========================
    # 2. 主力行为
    # =========================

    if main_force >= 80:
        score += 22
        reasons.append("主力行为强")
    elif main_force >= 60:
        score += 16
        reasons.append("主力行为活跃")
    elif main_force >= 30:
        score += 8
        reasons.append("主力行为初现")

    # =========================
    # 3. 模型辅助
    # =========================

    if model_score >= 60 and prob_up >= 0.6:
        score += 18
        reasons.append("模型强看多")
    elif model_score >= 30 and prob_up >= 0.3:
        score += 8
        reasons.append("模型轻度看多")

    if pred_label == 1:
        score += 6
        reasons.append("模型标签偏多")

    # =========================
    # 4. 放量结构
    # =========================

    if volume_ratio >= 3:
        score += 18
        reasons.append("爆量")
    elif volume_ratio >= 2:
        score += 13
        reasons.append("明显放量")
    elif volume_ratio >= 1.5:
        score += 7
        reasons.append("温和放量")

    if amount_ma20 >= 500_000_000:
        score += 8
        reasons.append("成交额充足")
    elif amount_ma20 >= 200_000_000:
        score += 4
        reasons.append("成交额尚可")

    # =========================
    # 5. 突破结构
    # =========================

    if break_60d == 1:
        score += 20
        reasons.append("突破60日高点")
    elif break_20d == 1:
        score += 14
        reasons.append("突破20日高点")

    # =========================
    # 6. 短线情绪
    # =========================

    if pct_change >= 9.5:
        score += 18
        reasons.append("涨停特征")
    elif pct_change >= 5:
        score += 12
        reasons.append("大阳线")
    elif pct_change >= 2:
        score += 5
        reasons.append("日内走强")

    if red_count_3d >= 3:
        score += 8
        reasons.append("三连阳")
    elif red_count_5d >= 4:
        score += 6
        reasons.append("五日多阳")

    if big_up_count_10d >= 2:
        score += 8
        reasons.append("近期多次大涨")

    if limit_like_count_20d >= 1:
        score += 8
        reasons.append("近期有涨停基因")

    if lower_shadow >= 0.03 and pct_change > 0:
        score += 6
        reasons.append("下影承接")

    if shrink_pullback == 1:
        score += 10
        reasons.append("缩量回踩")

    # =========================
    # 7. 低位启动加分
    # =========================

    if 0.25 <= position_60d <= 0.65 and pct_20d < 25:
        score += 10
        reasons.append("低中位启动")
    elif position_60d < 0.25 and pct_change > 2:
        score += 6
        reasons.append("低位异动")

    # =========================
    # 风险扣分
    # =========================

    if prob_down >= 0.55:
        risk += 18
        reasons.append("模型下跌概率偏高")

    if pred_label == -1:
        risk += 12
        reasons.append("模型标签偏空")

    if pct_change <= -3:
        risk += 25
        reasons.append("当日明显走弱")

    if upper_shadow >= 0.06:
        risk += 15
        reasons.append("上影线过长")

    if pct_5d >= 25:
        risk += 15
        reasons.append("5日涨幅过大")

    if pct_10d >= 45:
        risk += 20
        reasons.append("10日涨幅过大")

    if turnover >= 25:
        risk += 12
        reasons.append("换手过高")

    if position_60d >= 0.9 and pct_20d >= 40:
        risk += 18
        reasons.append("高位加速风险")

    final_score = max(0, min(score - risk, 100))

    if final_score >= 80 and risk <= 20:
        level = "S"
        action = "重点观察"
    elif final_score >= 65 and risk <= 30:
        level = "A"
        action = "观察"
    elif final_score >= 50:
        level = "B"
        action = "轻观察"
    else:
        level = "C"
        action = "忽略"

    return pd.Series(
        {
            "yaogu_v3_score": final_score,
            "yaogu_v3_raw_score": score,
            "yaogu_v3_risk": risk,
            "yaogu_v3_level": level,
            "yaogu_v3_action": action,
            "yaogu_v3_reason": "；".join(reasons),
        }
    )


def main():
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"找不到数据集：{DATASET_PATH}")

    print("读取训练数据集...")
    df = pd.read_csv(DATASET_PATH, encoding="utf-8-sig", dtype={"symbol": str})
    df["symbol"] = df["symbol"].astype(str).str.zfill(6)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    groups = []
    for symbol, g in df.groupby("symbol"):
        groups.append(add_short_features(g))

    df = pd.concat(groups, ignore_index=True)

    latest_signal = None
    if os.path.exists(LATEST_SIGNAL_PATH):
        latest_signal = pd.read_csv(LATEST_SIGNAL_PATH, encoding="utf-8-sig", dtype={"symbol": str})
        latest_signal["symbol"] = latest_signal["symbol"].astype(str).str.zfill(6)

        keep_cols = ["symbol", "prob_up", "prob_down", "model_score", "pred_label"]
        keep_cols = [c for c in keep_cols if c in latest_signal.columns]

        df = df.merge(
            latest_signal[keep_cols],
            on="symbol",
            how="left",
            suffixes=("", "_latest"),
        )

    else:
        df["prob_up"] = 0
        df["prob_down"] = 0
        df["model_score"] = 0
        df["pred_label"] = 0

    latest_date = df["date"].max()
    latest = df[df["date"] == latest_date].copy()

    print(f"最新日期：{latest_date}")
    print(f"最新股票数量：{len(latest)}")

    score_df = latest.apply(score_yaogu_v3, axis=1)
    result = pd.concat([latest, score_df], axis=1)

    output_cols = [
        "symbol",
        "date",
        "close",
        "pct_change",
        "turnover",
        "rainbow_score",
        "main_force_score",
        "model_score",
        "prob_up",
        "prob_down",
        "pred_label",
        "volume_ratio_v3",
        "pct_5d",
        "pct_10d",
        "pct_20d",
        "break_20d_high",
        "break_60d_high",
        "red_count_3d",
        "big_up_count_10d",
        "limit_like_count_20d",
        "position_60d",
        "yaogu_v3_score",
        "yaogu_v3_raw_score",
        "yaogu_v3_risk",
        "yaogu_v3_level",
        "yaogu_v3_action",
        "yaogu_v3_reason",
    ]

    output_cols = [c for c in output_cols if c in result.columns]

    result = result.sort_values(
        ["yaogu_v3_score", "yaogu_v3_raw_score"],
        ascending=False,
    )

    os.makedirs("output", exist_ok=True)
    result[output_cols].to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    strong = result[result["yaogu_v3_level"].isin(["S", "A"])]

    print("\n==============================")
    print("妖股模型 v3 扫描完成")
    print(f"输出文件：{OUTPUT_PATH}")
    print(f"S/A 候选数量：{len(strong)}")

    if len(strong) > 0:
        print("\nTop 候选：")
        print(strong[output_cols].head(20))
    else:
        print("\n今天没有 S/A 级候选，建议空仓观察。")


if __name__ == "__main__":
    main()


In [ ]:
!streamlit run app.py